In [0]:
%python
import re
from pyspark import pipelines as dp
from pyspark.sql import functions as F

excluded_volume_path = spark.conf.get("excluded_indiv_data_volue")
cms_prov_comp_volume_path = spark.conf.get("cms_prov_comp_volume_path")

def clean(name):
    return re.sub(r"[ ,;{}()=\n\t]+", "_", name.strip())

@dp.table(
    name="northstar_network_bronze.excluded_indiv_bronze.excluded_indiv_data_table",
    partition_cols=["STATE"],
    table_properties={
        "delta.appendOnly": "false",
        "pipelines.reset.allowed": "true"
    }
)
def ingest_excl_indiv_data():
    excl_indiv_df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaEvolutionMode", "rescue")  
        .load(excluded_volume_path)
        .withColumn("ingestion_date", F.current_date())
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("source_file_path", F.col("_metadata.file_path"))
        .withColumn("source_file_name", F.col("_metadata.file_name"))
    )
    return excl_indiv_df


@dp.table(
    name="northstar_network_bronze.cms_provider_compare.cms_prov_comp_ingest_data",
    partition_cols=["pri_spec"],
    table_properties={
        "delta.appendOnly": "false",
        "pipelines.reset.allowed": "true",
        "delta.columnMapping.mode": "name"
    }
)
def ingest_cms_prov_compare_data():
    prov_compare_data = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaEvolutionMode", "rescue")  
        .load(cms_prov_comp_volume_path)
    )
    clean_df = prov_compare_data.toDF(*[clean(c) for c in prov_compare_data.columns])
    
    return (
        clean_df.withColumn("ingestion_date", F.current_date())
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("source_file_path", F.col("_metadata.file_path"))
        .withColumn("source_file_name", F.col("_metadata.file_name"))
    )
